In [1]:
import json
import random
import pathlib

SEED   = 42
SPLIT  = 0.95

SOURCES = [
    ("data/bangla_chat.jsonl",  "bangla"),
    ("data/english_chat.jsonl", "english"),
    ("data/tool_calling.jsonl", "tool"),
]

In [2]:
def load_jsonl(path):
    samples = []
    with open(path, encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                samples.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  Skipping line {i} in {path}: {e}")
    return samples


def validate(sample):
    messages = sample.get("messages", [])
    if len(messages) < 2:
        return False
    if messages[-1].get("role") != "assistant":
        return False
    return any(
        msg.get("role") == "assistant" and (msg.get("content") or "").strip()
        for msg in messages
    )

In [3]:
random.seed(SEED)

all_samples = []
counts = {}

for path, source in SOURCES:
    p = pathlib.Path(path)
    if not p.exists():
        print(f"WARNING: {path} not found — skipping")
        continue

    raw     = load_jsonl(path)
    valid   = [s for s in raw if validate(s)]
    dropped = len(raw) - len(valid)

    counts[source] = len(valid)
    all_samples.extend(valid)

    print(f"{path:35s}  loaded={len(raw):>5}  valid={len(valid):>5}  dropped={dropped:>3}")

print(f"\nTotal before shuffle : {len(all_samples)}")

data/bangla_chat.jsonl               loaded= 8000  valid= 8000  dropped=  0
data/english_chat.jsonl              loaded= 5000  valid= 5000  dropped=  0
data/tool_calling.jsonl              loaded=  570  valid=  570  dropped=  0
data/tool_calling_test.jsonl         loaded=   55  valid=   55  dropped=  0

Total before shuffle : 13625


In [4]:
random.shuffle(all_samples)

split_idx  = int(len(all_samples) * SPLIT)
train_data = all_samples[:split_idx]
val_data   = all_samples[split_idx:]

output_dir = pathlib.Path("data")
output_dir.mkdir(exist_ok=True)

def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in data:
            f.write(json.dumps(s, ensure_ascii=False) + "\n")

save_jsonl(train_data, output_dir / "train.jsonl")
save_jsonl(val_data,   output_dir / "val.jsonl")

print(f"{'='*50}")
print(f"Bangla chat   : {counts.get('bangla',  0):>6}")
print(f"English chat  : {counts.get('english', 0):>6}")
print(f"Tool calling  : {counts.get('tool',    0):>6}")
print(f"{'─'*50}")
print(f"Total         : {len(all_samples):>6}")
print(f"Train (95%)   : {len(train_data):>6}  → data/train.jsonl")
print(f"Val   ( 5%)   : {len(val_data):>6}  → data/val.jsonl")
print(f"{'='*50}")

Bangla chat   :   8000
English chat  :   5000
Tool calling  :     55
──────────────────────────────────────────────────
Total         :  13625
Train (95%)   :  12943  → data/train.jsonl
Val   ( 5%)   :    682  → data/val.jsonl
